In [ ]:
using Gridap        
using GridapGmsh
using Gridap.Geometry
using Gridap.TensorValues
using Plots
using LinearAlgebra
using  Gridap.Fields
using  Gridap.CellData
using  Gridap.ReferenceFEs  
using  Gridap.Fields

In [ ]:
const I2 = TensorValue(1.0,0.0,0.0,1.0)
const I1 = VectorValue(0.0,1.0)
const DegCase = 3   # 1 = sym part of energy, 2 Total energy with out curvature part, 3 Total Energy
const p = 0.5

In [ ]:
model = GmshDiscreteModel("ThreePointBending.msh")
writevtk(model,"ThreePointBending")

In [ ]:
labels = get_face_labeling(model)

In [ ]:
const ν = 0.2
const  E = 19000
const G = E/(2*(1+ν))

In [ ]:
const lsp = 3.0
const Gc = 0.113
const η = 1e-10

In [ ]:
deg_fun(d) = d*d/(d*d+m*(1-d)*(1+p*(1-d)))   # Written such that d = 0 (damage)
g_dash_s_fun(ϕ) = ((ϕ-1)*(ϕ*(2*p+1)+1)*m)/((ϕ*ϕ*(m*p+1)+(ϕ*(m-2)+1))*((ϕ*ϕ*(m*p+1)+(ϕ*(m-2)+1))))

In [ ]:
const λ_ps = (E*ν)/((1+ν)*(1-2*ν)) # plane strain
const μ = G
const λ = λ_ps*(2*μ/(λ_ps+2*μ))
const fₜ = 2.4
ψ_crit = 0.00019
const F_crit = ψ_crit*lsp/(Gc)
const m = 3/(8*F_crit)

In [ ]:
const κ = λ + μ
const lb = 1.0
const γ = 4*μ*lb*lb

In [ ]:
const N = 0.6
const μc = μ*N^2/(1 - N^2)

In [ ]:
function σ(ε)
λ*tr(ε)*one(ε) + 2*μ*ε
end

function σ_mod(ε,s_in)
(deg_fun(s_in)+η)*σ(ε)
end

In [ ]:
function project(q,model,dΩ,order)
  reffe = ReferenceFE(lagrangian,Float64,order)
  V = FESpace(model,reffe,conformity=:L2)
  a(u,v) = ∫( u*v )*dΩ
  l(v) = ∫( v*q )*dΩ
  op = AffineFEOperator(a,l,V,V)
  qh = solve(op)
  qh
end

In [ ]:
function project_vector(q,model,dΩ,order)
  reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
  V = FESpace(model,reffe,conformity=:L2)
  a(u,v) = ∫( u⊙v )*dΩ
  l(v) = ∫( v⊙q)*dΩ
  op = AffineFEOperator(a,l,V,V)
  qh = solve(op)
  qh
end

In [ ]:
function project_tensor(q,model,dΩ,order)
  reffe = ReferenceFE(lagrangian,TensorValue{2,2,Float64,4},order)
  V = FESpace(model,reffe,conformity=:L2)
  a(u,v) = ∫( u⊙v )*dΩ
  l(v) = ∫( v⊙q)*dΩ
  op = AffineFEOperator(a,l,V,V)
  qh = solve(op)
  qh
end

In [ ]:
order = 1
degree = 2*order

In [ ]:
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)

In [ ]:
LoadTagId = get_tag_from_name(labels,"LoadEdge")
Γ_Load = BoundaryTriangulation(model,tags = LoadTagId)
dΓ_Load = Measure(Γ_Load,degree)
n_Γ_Load = get_normal_vector(Γ_Load)

In [ ]:
reffe_PF = ReferenceFE(lagrangian,Float64,order)
V0_PF = TestFESpace(model,reffe_PF;
  conformity=:H1)
U_PF = TrialFESpace(V0_PF)
sh = zero(V0_PF)

In [ ]:
reffe_Disp = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
V0_Disp = TestFESpace(model,reffe_Disp;
          conformity=:H1,
          dirichlet_tags=["LeftEdge","RightEdge","LoadEdge"],
          dirichlet_masks=[(true,true),(false,true),(false,true)])
uh = zero(V0_Disp)

In [ ]:
function  new_EnergyState(ψPlusPrev_in,ψhPos_in)
    ψPlus_in = F_crit + F_crit*0.5*(ψhPos_in/F_crit-1 + abs(ψhPos_in/F_crit-1)) 
    if ψPlus_in  >= ψPlusPrev_in
            ψPlus_out = ψPlus_in 
        else
            ψPlus_out = ψPlusPrev_in
    end
    true,ψPlus_out
end

In [ ]:
using Gridap.MultiField

In [ ]:
function Q(θ_in,Q_init)        
Q_upd = exp_map∘(skew(θ_in))⋅Q_init
Q_new = project_tensor(Q_upd,model,dΩ,order)
 return Q_new   
end

In [ ]:
function ψPos(ε) 
    Pos_Tr_E = 0.5*(tr(ε) + abs(tr(ε)))
    ε_dev = ε - 0.5*tr(ε)*I2
    ψPlus = (lsp/Gc)*(0.5*κ*Pos_Tr_E*Pos_Tr_E + μ*(ε_dev⊙ε_dev))
    return ψPlus
end

In [ ]:
function   stepDisp(sh_in,uApp)
uApp1(x) = VectorValue(0.0,0.0)
uApp2(x) = VectorValue(0.0,0.0)  
uApp3(x) = VectorValue(0.0,-uApp)
U_Disp = TrialFESpace(V0_Disp,[uApp1,uApp2,uApp3])
a_disp(u,v) = ∫( ε(v) ⊙ ((deg_fun(sh_in))*((σ∘(ε(u)) ))))*dΩ
l_disp(v) = 0

op = AffineFEOperator(a_disp,l_disp,U_Disp,V0_Disp)
uh_out = solve(op)
        return uh_out  
end

In [ ]:
function  stepPhaseField(ψPlusPrev_in,s_in,cache)
res_PF(s,ϕ) = ∫( (3/4)*lsp*lsp*∇(ϕ)⋅∇(s) - (g_dash_s_fun(1-s))*ψPlusPrev_in*ϕ - (3/8)*ϕ )*dΩ  
op_PF = FEOperator(res_PF,U_PF,V0_PF)
nls = NLSolver(
show_trace=true, method=:newton, iterations = 30, linesearch=BackTracking())
solver = FESolver(nls)

sh_out = FEFunction(U_PF,s_in)
sh_out, cache = solve!(sh_out,solver,op_PF,cache)
  return sh_out, get_free_dof_values(sh_out), cache
end

In [ ]:
using LineSearches: BackTracking

In [ ]:
uApp = 0.0
delu = 1e-3
uAppMax = 0.6
const innerMax = 100
count = 0
sh_in = ones(Float64,num_free_dofs(V0_PF))
sPrev = CellState(1.0,dΩ)
sh = project(sPrev,model,dΩ,order)
cache = nothing
uh_in = get_free_dof_values(uh) 
ψPlusPrev = CellState(F_crit,dΩ)
Load = Float64[]
Displacement = Float64[]
uh_in_FE = uh
start_time=time()
while uApp .< uAppMax 

    uApp = uApp .+ delu  
    count = count .+ 1
    print("\n Entering displacemtent step :", float(uApp))
    
       
   for inner = 1:innerMax
        ψhPlusPrev = project(ψPlusPrev,model,dΩ,order)
        e = uh - uh_in_FE
         err = sqrt(sum( ∫( e⊙e )*dΩ ))
        print("\n error = ",float(err))
        uh_in_FE = uh
        sh,sh_in,cache = stepPhaseField(ψhPlusPrev,sh_in,cache) 
        uh = stepDisp(sh,uApp)
        ψhPos_in = ψPos∘(ε(uh))
        update_state!(new_EnergyState,ψPlusPrev,ψhPos_in) 
        if err < 1e-8
            break 
        end  
    end
    Node_Force = sum(∫(n_Γ_Load⋅(σ_mod∘(ε(uh),sh)))dΓ_Load)
    push!(Load, abs(Node_Force[2]))
    push!(Displacement, uApp)
    
     if mod(count,5)  == 0
        ψhPlusPrev = project(ψPlusPrev,model,dΩ,order)
writevtk(Ω,"results_PhaseField$count",cellfields=
        ["uh"=>uh ,"sigma"=>σ∘(ε(uh)),"s"=>sh,"en"=>ψhPlusPrev])
    end
end
end_time=time()
elapsed_time=end_time-start_time

In [ ]:
plot(Displacement,Load)

In [ ]:
using DelimitedFiles
Disp = writedlm("ThreePointBend1emin3.csv",  [Displacement Load], ',')